In [1]:
#simulate_wearable_data.py

import pandas as pd
import time
import random
import os
import json

# --- Configuration ---
DATA_SOURCE_PATH = "/Users/bhanutejamalineni/phis_project/data/processed/simulation_data.csv" 
SIMULATION_INTERVAL_SECONDS = 3 
MAX_VARIABILITY = 0.01 
SIMULATE_FOR_ATHLETE = 'ATH001' 

# --- Function to generate simulated data ---
def generate_simulated_data(df, start_index=0):
    """
    Generator that yields one data point from the DataFrame at a time,
    with added minor random variability to numerical fields.
    """
    for i in range(start_index, len(df)):
        row = df.iloc[i].to_dict()

        for key, value in row.items():
            if isinstance(value, (int, float)) and key not in ['timestamp', 'Athlete_ID']:
                variability = 1 + (random.uniform(-MAX_VARIABILITY, MAX_VARIABILITY))
                row[key] = value * variability
                # Ensure values stay positive and within reasonable bounds
                if key == 'heart_rate':
                    row[key] = max(40, min(200, int(row[key]))) # Common resting to max HR
                elif key == 'steps':
                    row[key] = max(0, int(row[key])) # Steps can't be negative
                elif key == 'body_temperature':
                    row[key] = round(max(35.0, min(40.0, row[key])), 1) # Human body temp range
                elif key == 'blood_oxygen':
                    row[key] = max(85, min(100, int(row[key]))) # Blood oxygen range
                    
        if 'timestamp' in row and isinstance(row['timestamp'], pd.Timestamp):
            row['timestamp'] = row['timestamp'].isoformat()

        yield row
        time.sleep(SIMULATION_INTERVAL_SECONDS) # Pause to simulate real-time data
        
# --- Main simulation logic ---
if __name__ == "__main__":
    print(f"Starting wearable data simulation. Data will be printed every {SIMULATION_INTERVAL_SECONDS} seconds.")
    print("Press Ctrl+C to stop the simulation.")
    try:
        combined_df_for_sim = pd.read_csv(DATA_SOURCE_PATH, parse_dates=['timestamp'])
        print(f"Loaded preprocessed data from {DATA_SOURCE_PATH}")
    except FileNotFoundError:
        print(f"FATAL ERROR: Preprocessed data not found at {DATA_SOURCE_PATH}.")
        print("Please run the Jupyter Notebook preprocessing step first to create this file.")
        exit()
    except Exception as e:
        print(f"FATAL ERROR: Could not load data from {DATA_SOURCE_PATH}: {e}")
        exit()

    if SIMULATE_FOR_ATHLETE and 'Athlete_ID' in combined_df_for_sim.columns:
        initial_rows = len(combined_df_for_sim)
        combined_df_for_sim = combined_df_for_sim[combined_df_for_sim['Athlete_ID'] == SIMULATE_FOR_ATHLETE].copy()
        if combined_df_for_sim.empty:
            print(f"Warning: No data found for Athlete_ID '{SIMULATE_FOR_ATHLETE}'. Simulating all available data instead.")
        else:
            print(f"Filtered data to simulate only for Athlete_ID: {SIMULATE_FOR_ATHLETE} ({len(combined_df_for_sim)} rows out of {initial_rows})")
    elif SIMULATE_FOR_ATHLETE:
        print(f"Warning: 'Athlete_ID' column not found, cannot filter for '{SIMULATE_FOR_ATHLETE}'. Simulating all data.")



    if 'timestamp' in combined_df_for_sim.columns:
        combined_df_for_sim['timestamp'] = pd.to_datetime(combined_df_for_sim['timestamp'])
    else:
        print("FATAL ERROR: 'timestamp' column is missing in the simulation data after loading.")
        exit()

    try:
        data_generator = generate_simulated_data(combined_df_for_sim)
        counter = 0
        for data_point in data_generator:
            print(f"[{counter + 1}] Simulated Data Point: {json.dumps(data_point)}")
            counter += 1
            if counter >= len(combined_df_for_sim): 
                print("End of dataset reached. Simulation complete.")
                break

    except KeyboardInterrupt:
        print("\nSimulation stopped by user.")
    except Exception as e:
        print(f"\nAn error occurred during simulation: {e}")

Starting wearable data simulation. Data will be printed every 3 seconds.
Press Ctrl+C to stop the simulation.
Loaded preprocessed data from /Users/bhanutejamalineni/phis_project/data/processed/simulation_data.csv
Filtered data to simulate only for Athlete_ID: ATH001 (43 rows out of 500)
[1] Simulated Data Point: {"timestamp": "2025-04-10T09:10:00", "Athlete_ID": "ATH001", "heart_rate": 130, "steps": 851, "body_temperature": 36.8, "blood_pressure": "116/90", "blood_oxygen": 99, "activity_status": "Resting"}
[2] Simulated Data Point: {"timestamp": "2025-04-10T09:15:00", "Athlete_ID": "ATH001", "heart_rate": 155, "steps": 730, "body_temperature": 37.2, "blood_pressure": "135/75", "blood_oxygen": 100, "activity_status": "Cycling"}
[3] Simulated Data Point: {"timestamp": "2025-04-10T09:50:00", "Athlete_ID": "ATH001", "heart_rate": 89, "steps": 707, "body_temperature": 36.6, "blood_pressure": "136/71", "blood_oxygen": 96, "activity_status": "Cycling"}
[4] Simulated Data Point: {"timestamp": 